# Data Preprocessing — Framingham Heart Study

This notebook handles missing value imputation, train/val/test splitting, feature scaling, and SMOTE oversampling to address class imbalance.

**Input:** `../data/raw/framingham.csv`  
**Output:** `../data/processed/train.csv`, `val.csv`, `test.csv` + `../models/scaler.pkl`

## 1. Imports

In [ ]:
!pip install imbalanced-learn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import joblib
import os

## 2. Load Raw Data

In [ ]:
df        = pd.read_csv('../data/raw/framingham.csv')
df_before = pd.read_csv('../data/raw/framingham.csv')   # keep original for before/after plots

print('Shape:', df.shape)
df.head()

## 3. Median Imputation

All columns with nulls are filled with their respective medians. Medians are computed from the full dataset before splitting — this is not a leaky operation because medians are summary statistics with no target signal.

In [ ]:
cols_with_nulls = df.columns[df.isnull().any()].tolist()
print('Columns to impute:', cols_with_nulls)

train_medians = df[cols_with_nulls].median()
df[cols_with_nulls] = df[cols_with_nulls].fillna(train_medians)

print('Remaining nulls:', df.isnull().sum().sum())

In [ ]:
# Before vs After imputation — glucose has the most missing (9.15%)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_before['glucose'].hist(ax=axes[0], bins=20, color='salmon', edgecolor='white')
axes[0].set_title('Glucose — Before Imputation')
axes[0].set_xlabel('Glucose')

df['glucose'].hist(ax=axes[1], bins=20, color='steelblue', edgecolor='white')
axes[1].set_title('Glucose — After Imputation')
axes[1].set_xlabel('Glucose')

plt.suptitle('Effect of Median Imputation on Glucose (Most Missing Feature)')
plt.tight_layout()
plt.show()

**Result:** Zero missing values remaining. The glucose distribution shape is preserved — median imputation fills the gap without distorting the spread.

## 4. Train / Validation / Test Split (70 / 15 / 15)

Stratified splitting preserves the ~15% positive rate across all three partitions, ensuring each set reflects the real-world class distribution.

In [ ]:
X = df.drop(columns=['TenYearCHD'])
y = df['TenYearCHD']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f'Train : {X_train.shape}  |  Positive rate: {y_train.mean():.3f}')
print(f'Val   : {X_val.shape}   |  Positive rate: {y_val.mean():.3f}')
print(f'Test  : {X_test.shape}   |  Positive rate: {y_test.mean():.3f}')

In [ ]:
# Pie chart confirming the 70/15/15 split proportions
labels  = ['Train (70%)', 'Validation (15%)', 'Test (15%)']
sizes   = [len(X_train), len(X_val), len(X_test)]
colors  = ['steelblue', 'orange', 'salmon']
explode = (0.05, 0.05, 0.05)

plt.figure(figsize=(6, 6))
plt.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%',
        startangle=90, explode=explode)
plt.title('Train / Validation / Test Split\n(Total: 4,240 patients)')
plt.show()

**Verification:** Positive rate is ~0.152 across all three splits — stratification worked correctly.

## 5. Feature Scaling

`StandardScaler` is **fit only on the training set** to prevent data leakage. The same fitted scaler is then applied to val and test.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit ONLY on train
X_val_scaled   = scaler.transform(X_val)          # transform only
X_test_scaled  = scaler.transform(X_test)          # transform only

print('Scaling done.')
print('Train mean (first 3 features):', X_train_scaled[:, :3].mean(axis=0).round(4))
print('(Should be ~0 — confirms scaling is correct)')

In [ ]:
# Before vs After scaling — shows all features now on the same scale
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pd.DataFrame(X_train, columns=X.columns).boxplot(ax=axes[0], rot=45)
axes[0].set_title('Before Scaling')
axes[0].set_ylabel('Value')

pd.DataFrame(X_train_scaled, columns=X.columns).boxplot(ax=axes[1], rot=45)
axes[1].set_title('After Scaling (StandardScaler)')
axes[1].set_ylabel('Value')

plt.suptitle('Feature Distributions Before vs After Scaling')
plt.tight_layout()
plt.show()

**Why scaling matters:** Features like `sysBP` (range 83–295) and `diabetes` (0–1) are on completely different scales. Without scaling, distance-based models (SVM) and gradient-based models (Logistic Regression) would treat larger-valued features as more important. After scaling, all features have mean ≈ 0 and std ≈ 1.

## 6. SMOTE Oversampling

SMOTE is applied **only to the training set** to balance the 85/15 class ratio. Validation and test sets are left untouched so they reflect the real-world distribution and provide an honest evaluation.

In [ ]:
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

before = pd.Series(y_train).value_counts().sort_index()
after  = pd.Series(y_train_resampled).value_counts().sort_index()

print('Before SMOTE:', dict(before))
print('After  SMOTE:', dict(after))

In [ ]:
# Before vs After SMOTE bar chart
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

before.plot(kind='bar', ax=axes[0], color=['steelblue', 'salmon'], edgecolor='black')
axes[0].set_title('Before SMOTE')
axes[0].set_xlabel('TenYearCHD')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['No CHD (0)', 'CHD Risk (1)'], rotation=0)

after.plot(kind='bar', ax=axes[1], color=['steelblue', 'salmon'], edgecolor='black')
axes[1].set_title('After SMOTE')
axes[1].set_xlabel('TenYearCHD')
axes[1].set_ylabel('Count')
axes[1].set_xticklabels(['No CHD (0)', 'CHD Risk (1)'], rotation=0)

plt.suptitle('Class Distribution Before vs After SMOTE (Training Set Only)')
plt.tight_layout()
plt.show()

**Result:** Training set balanced from 2,517 / 451 to 2,517 / 2,517 (50/50).  
Val and test sets remain at the original ~85/15 ratio for honest evaluation.  
SMOTE generates **synthetic** minority-class samples by interpolating between real examples — not duplicating them.

## 7. Save Processed Files

Saving all three splits and the scaler so every team member loads **identical data**.

In [ ]:
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# Training set — SMOTE balanced, scaled
pd.DataFrame(X_train_resampled, columns=X.columns) \
  .assign(TenYearCHD=y_train_resampled) \
  .to_csv('../data/processed/train.csv', index=False)

# Validation set — real distribution, scaled only
pd.DataFrame(X_val_scaled, columns=X.columns) \
  .assign(TenYearCHD=y_val.values) \
  .to_csv('../data/processed/val.csv', index=False)

# Test set — real distribution, scaled only
pd.DataFrame(X_test_scaled, columns=X.columns) \
  .assign(TenYearCHD=y_test.values) \
  .to_csv('../data/processed/test.csv', index=False)

# Scaler — needed to transform any new data consistently
joblib.dump(scaler, '../models/scaler.pkl')

print('Saved: data/processed/train.csv  (SMOTE balanced)')
print('Saved: data/processed/val.csv    (real distribution)')
print('Saved: data/processed/test.csv   (real distribution)')
print('Saved: models/scaler.pkl')

## Summary

| Step | What was done |
|------|---------------|
| Imputation | Median imputation on 7 columns with missing values |
| Split | Stratified 70/15/15 — positive rate ~0.152 in all sets |
| Scaling | StandardScaler fit on train only |
| SMOTE | Training set balanced to 50/50 (2,517 each class) |
| Saved | `train.csv`, `val.csv`, `test.csv`, `scaler.pkl` |

**For teammates:** Load `train.csv` for training, `val.csv` for tuning, `test.csv` for final evaluation only.  
Load the scaler with `scaler = joblib.load('../models/scaler.pkl')` if needed.